In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import ParameterGrid

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

RANDOM_STATE = 42
TARGET_COL = "Is1Winner"

In [2]:
TRAIN_PATH = "../tourney_train_m.csv"
TEST_PATH  = "../tourney_test_m.csv"
VAL_PATH   = "../tourney_val_m.csv"

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df  = pd.read_csv(TEST_PATH, low_memory=False)
val_df   = pd.read_csv(VAL_PATH, low_memory=False)

print("train:", train_df.shape)
print("test :", test_df.shape)
print("val  :", val_df.shape)

print()
print("Target in train:", TARGET_COL in train_df.columns)
print("Target in test :", TARGET_COL in test_df.columns)
print("Target in val  :", TARGET_COL in val_df.columns)

train: (1315, 175)
test : (67, 175)
val  : (67, 175)

Target in train: True
Target in test : True
Target in val  : True


In [3]:
manual_drop_cols = [
    "Season",
    "1TeamID",
    "2TeamID",
    "TeamID_2"
]

manual_drop_cols = [c for c in manual_drop_cols if c in train_df.columns]
manual_drop_cols

['Season', '1TeamID', '2TeamID', 'TeamID_2']

In [4]:
def build_feature_sets(train_df, test_df, val_df, target_col, manual_drop_cols):
    drop_cols = set(manual_drop_cols + [target_col])

    base_cols = [c for c in train_df.columns if c not in drop_cols]

    # RAW
    X_train_raw = train_df[base_cols].copy()
    X_test_raw  = test_df[base_cols].copy()
    X_val_raw   = val_df[base_cols].copy()

    # DIFF
    cols_2 = [c for c in base_cols if c.endswith("_2")]
    cols_1 = [c[:-2] for c in cols_2 if c[:-2] in base_cols]

    X_train_diff = pd.DataFrame(index=train_df.index)
    X_test_diff  = pd.DataFrame(index=test_df.index)
    X_val_diff   = pd.DataFrame(index=val_df.index)

    for c in cols_1:
        X_train_diff[f"{c}_diff"] = train_df[c] - train_df[f"{c}_2"]
        X_test_diff[f"{c}_diff"]  = test_df[c] - test_df[f"{c}_2"]
        X_val_diff[f"{c}_diff"]   = val_df[c] - val_df[f"{c}_2"]

    # RAW + DIFF
    X_train_raw_diff = pd.concat([X_train_raw, X_train_diff], axis=1)
    X_test_raw_diff  = pd.concat([X_test_raw, X_test_diff], axis=1)
    X_val_raw_diff   = pd.concat([X_val_raw, X_val_diff], axis=1)

    return {
        "raw": (X_train_raw, X_test_raw, X_val_raw),
        "diff": (X_train_diff, X_test_diff, X_val_diff),
        "raw_diff": (X_train_raw_diff, X_test_raw_diff, X_val_raw_diff)
    }
    
def clean_and_impute(X_train, X_test, X_val, missing_threshold=0.60, corr_threshold=0.995):
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_val = X_val.copy()

    # kolumny całkiem puste
    all_nan_cols = [c for c in X_train.columns if X_train[c].isna().all()]

    # zbyt duży missing
    high_missing_cols = [c for c in X_train.columns if X_train[c].isna().mean() > missing_threshold]

    # stałe
    nunique = X_train.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()

    drop_cols = sorted(set(all_nan_cols + high_missing_cols + constant_cols))

    X_train = X_train.drop(columns=drop_cols, errors="ignore")
    X_test  = X_test.drop(columns=drop_cols, errors="ignore")
    X_val   = X_val.drop(columns=drop_cols, errors="ignore")

    # imputacja
    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)
    X_val   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)

    # korelacje
    if X_train.shape[1] > 1:
        corr = X_train.corr(numeric_only=True).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_cols = [c for c in upper.columns if any(upper[c] > corr_threshold)]
    else:
        high_corr_cols = []

    X_train = X_train.drop(columns=high_corr_cols, errors="ignore")
    X_test  = X_test.drop(columns=high_corr_cols, errors="ignore")
    X_val   = X_val.drop(columns=high_corr_cols, errors="ignore")

    meta = {
        "dropped_bad_cols": drop_cols,
        "dropped_high_corr_cols": high_corr_cols
    }

    return X_train, X_test, X_val, meta

In [5]:
def plot_corr(df, title="Correlation Matrix", figsize=(20, 16), annot=False):
    corr = df.corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        corr,
        mask=~mask,
        cmap="RdBu_r",
        center=0,
        vmin=-1, vmax=1,
        annot=annot,
        fmt=".2f",
        linewidths=0.5,
        square=True,
        cbar_kws={"shrink": 0.8, "label": "Correlation"},
        ax=ax
    )
    ax.set_title(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [6]:
y_train = train_df[TARGET_COL].astype(int).copy()
y_test  = test_df[TARGET_COL].astype(int).copy()
y_val   = val_df[TARGET_COL].astype(int).copy()

print(f"{len(y_train)},{len(y_test)},{len(y_val)}")


feature_sets = build_feature_sets(train_df, test_df, val_df, TARGET_COL, manual_drop_cols)

for name, (Xtr, Xte, Xva) in feature_sets.items():
    print(name, Xtr.shape, Xte.shape, Xva.shape)

1315,67,67
raw (1315, 170) (67, 170) (67, 170)
diff (1315, 85) (67, 85) (67, 85)
raw_diff (1315, 255) (67, 255) (67, 255)


In [7]:
# przetestować przy wyżuceniu większej ilości kolumn
prepared_feature_sets = {}

for feat_name, (Xtr, Xte, Xva) in feature_sets.items():
    Xtr2, Xte2, Xva2, meta = clean_and_impute(Xtr, Xte, Xva, corr_threshold=0.99)
    prepared_feature_sets[feat_name] = {
        "X_train": Xtr2,
        "X_test": Xte2,
        "X_val": Xva2,
        "meta": meta
    }
    print(f"{feat_name}: {Xtr2.shape}, {Xte2.shape}, {Xva2.shape}")

raw: (1315, 166), (67, 166), (67, 166)
diff: (1315, 83), (67, 83), (67, 83)
raw_diff: (1315, 249), (67, 249), (67, 249)


In [8]:
def evaluate_proba(y_true, proba, threshold=0.5):
    proba = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    pred = (proba >= threshold).astype(int)

    return {
        "brier": brier_score_loss(y_true, proba),
        "logloss": log_loss(y_true, proba),
        "auc": roc_auc_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "pred_mean": float(np.mean(proba))
    }


def print_metric_delta(train_metrics, test_metrics):
    print("TRAIN brier:", round(train_metrics["brier"], 6))
    print("TEST  brier:", round(test_metrics["brier"], 6))
    print("DELTA brier:", round(test_metrics["brier"] - train_metrics["brier"], 6))


def hard_clip_extreme(proba, high=0.85, low=0.45):
    p = np.asarray(proba).copy()
    p[p >= high] = 1.0
    p[p <= low] = 0.0
    return p


def power_sharpen(proba, alpha=1.10):
    p = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    num = np.power(p, alpha)
    den = num + np.power(1 - p, alpha)
    return num / den


def temperature_sharpen(proba, temperature=0.92):
    p = np.clip(np.asarray(proba), 1e-8, 1 - 1e-8)
    logits = np.log(p / (1 - p))
    logits = logits / temperature
    out = 1 / (1 + np.exp(-logits))
    return np.clip(out, 1e-8, 1 - 1e-8)

In [9]:
def make_model(model_name, params):
    if model_name == "extratrees":
        return ExtraTreesClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_name == "lightgbm":
        return LGBMClassifier(
            objective="binary",
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1,
            **params
        )

    if model_name == "xgboost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            **params
        )

    if model_name == "catboost":
        return CatBoostClassifier(
            loss_function="Logloss",
            verbose=False,
            random_seed=RANDOM_STATE,
            **params
        )

    if model_name == "histgb":
        return HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **params
        )

    if model_name == "logreg":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(random_state=RANDOM_STATE, **params))
        ])

    if model_name == "svm":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(random_state=RANDOM_STATE, **params))
        ])
    
    if model_name == "mlp":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(random_state=RANDOM_STATE, **params))
        ])

    raise ValueError(f"Nieznany model: {model_name}")

param_spaces = {
    "extratrees": [
    {"n_estimators": 500, "max_depth": 4,  "min_samples_leaf": 30, "max_features": 0.2, "criterion": "gini"},
    {"n_estimators": 800, "max_depth": 6,  "min_samples_leaf": 20, "max_features": 0.3, "criterion": "entropy"},
    {"n_estimators": 1000, "max_depth": 5, "min_samples_leaf": 25, "max_features": "sqrt", "criterion": "log_loss"},
],

"lightgbm": [
    {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 8,  "max_depth": 3, "min_child_samples": 50, "subsample": 0.6, "colsample_bytree": 0.4, "reg_alpha": 3.0, "reg_lambda": 10.0},
    {"n_estimators": 400, "learning_rate": 0.03, "num_leaves": 12, "max_depth": 4, "min_child_samples": 40, "subsample": 0.7, "colsample_bytree": 0.5, "reg_alpha": 2.0, "reg_lambda": 5.0},
    {"n_estimators": 600, "learning_rate": 0.02, "num_leaves": 15, "max_depth": 4, "min_child_samples": 30, "subsample": 0.7, "colsample_bytree": 0.5, "reg_alpha": 1.0, "reg_lambda": 5.0},
],

"xgboost": [
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "min_child_weight": 20, "subsample": 0.6, "colsample_bytree": 0.4, "gamma": 1.0, "reg_alpha": 3.0, "reg_lambda": 10.0},
    {"n_estimators": 400, "learning_rate": 0.03, "max_depth": 3, "min_child_weight": 15, "subsample": 0.7, "colsample_bytree": 0.5, "gamma": 0.5, "reg_alpha": 2.0, "reg_lambda": 5.0},
    {"n_estimators": 600, "learning_rate": 0.02, "max_depth": 3, "min_child_weight": 10, "subsample": 0.7, "colsample_bytree": 0.5, "gamma": 0.3, "reg_alpha": 1.0, "reg_lambda": 5.0},
],

"catboost": [
    {"iterations": 300, "learning_rate": 0.05, "depth": 3, "l2_leaf_reg": 20.0, "random_strength": 3.0, "min_data_in_leaf": 30},
    {"iterations": 500, "learning_rate": 0.03, "depth": 4, "l2_leaf_reg": 15.0, "random_strength": 2.0, "min_data_in_leaf": 20},
    {"iterations": 800, "learning_rate": 0.02, "depth": 4, "l2_leaf_reg": 10.0, "random_strength": 2.0, "min_data_in_leaf": 15},
],

"histgb": [
    {"learning_rate": 0.05, "max_iter": 200, "max_depth": 3, "min_samples_leaf": 40, "l2_regularization": 10.0, "max_leaf_nodes": 8},
    {"learning_rate": 0.03, "max_iter": 400, "max_depth": 4, "min_samples_leaf": 30, "l2_regularization": 5.0, "max_leaf_nodes": 12},
    {"learning_rate": 0.02, "max_iter": 600, "max_depth": 4, "min_samples_leaf": 20, "l2_regularization": 5.0, "max_leaf_nodes": 15},
],
    
    "logreg": [
        {"C": 0.01, "penalty": "l2", "solver": "lbfgs", "max_iter": 2000},
        {"C": 0.1,  "penalty": "l2", "solver": "lbfgs", "max_iter": 2000},
        {"C": 1.0,  "penalty": "l2", "solver": "lbfgs", "max_iter": 2000},
        {"C": 0.1,  "penalty": "l1", "solver": "saga",  "max_iter": 2000},
        {"C": 0.05, "penalty": "elasticnet", "solver": "saga", "l1_ratio": 0.5, "max_iter": 2000},
    ],
    "svm": [
        {"C": 0.1,  "kernel": "rbf", "gamma": "scale", "probability": True},
        {"C": 1.0,  "kernel": "rbf", "gamma": "scale", "probability": True},
        {"C": 10.0, "kernel": "rbf", "gamma": "auto",  "probability": True},
    ],
    "mlp": [
        {"hidden_layer_sizes": (64, 32),     "alpha": 1.0,  "learning_rate_init": 0.001, "max_iter": 1000, "early_stopping": True},
        {"hidden_layer_sizes": (128, 64, 32), "alpha": 2.0,  "learning_rate_init": 0.001, "max_iter": 1000, "early_stopping": True},
        {"hidden_layer_sizes": (32, 16),      "alpha": 5.0,  "learning_rate_init": 0.0005, "max_iter": 1500, "early_stopping": True},
    ],
}


In [10]:
search_rows = []
search_results = {}

for feat_name, feat_data in prepared_feature_sets.items():
    Xtr = feat_data["X_train"]
    Xte = feat_data["X_test"]
    Xva = feat_data["X_val"]

    print("=" * 100)
    print("FEATURE SET:", feat_name, Xtr.shape)

    for model_name, grid in param_spaces.items():
        print("-" * 80)
        print("MODEL:", model_name)

        best_test_brier = np.inf
        best_payload = None

        for i, params in enumerate(grid, start=1):
            model = make_model(model_name, params)
            model.fit(Xtr, y_train)

            train_proba = model.predict_proba(Xtr)[:, 1]
            test_proba  = model.predict_proba(Xte)[:, 1]
            val_proba   = model.predict_proba(Xva)[:, 1]

            train_metrics = evaluate_proba(y_train, train_proba)
            test_metrics  = evaluate_proba(y_test, test_proba)
            val_metrics   = evaluate_proba(y_val, val_proba)

            row = {
                "feature_set": feat_name,
                "model": model_name,
                "variant_no": i,
                "params": params,
                "train_brier": train_metrics["brier"],
                "test_brier": test_metrics["brier"],
                "val_brier": val_metrics["brier"],
                "train_auc": train_metrics["auc"],
                "test_auc": test_metrics["auc"],
                "val_auc": val_metrics["auc"],
                "brier_gap_test_minus_train": test_metrics["brier"] - train_metrics["brier"],
                "brier_gap_val_minus_train": val_metrics["brier"] - train_metrics["brier"]
            }
            search_rows.append(row)

            if test_metrics["brier"] < best_test_brier:
                best_test_brier = test_metrics["brier"]
                best_payload = {
                    "model": model,
                    "params": params,
                    "train_proba": train_proba,
                    "test_proba": test_proba,
                    "val_proba": val_proba,
                    "train_metrics": train_metrics,
                    "test_metrics": test_metrics,
                    "val_metrics": val_metrics,
                    "Xtr": Xtr,
                    "Xte": Xte,
                    "Xva": Xva
                }

        search_results[(feat_name, model_name)] = best_payload
        print("BEST TEST BRIER:", round(best_payload["test_metrics"]["brier"], 6))
        print("TRAIN/TEST delta:")
        print_metric_delta(best_payload["train_metrics"], best_payload["test_metrics"])

FEATURE SET: raw (1315, 166)
--------------------------------------------------------------------------------
MODEL: extratrees
BEST TEST BRIER: 0.205244
TRAIN/TEST delta:
TRAIN brier: 0.162641
TEST  brier: 0.205244
DELTA brier: 0.042603
--------------------------------------------------------------------------------
MODEL: lightgbm
BEST TEST BRIER: 0.223704
TRAIN/TEST delta:
TRAIN brier: 0.124911
TEST  brier: 0.223704
DELTA brier: 0.098793
--------------------------------------------------------------------------------
MODEL: xgboost
BEST TEST BRIER: 0.217888
TRAIN/TEST delta:
TRAIN brier: 0.159766
TEST  brier: 0.217888
DELTA brier: 0.058122
--------------------------------------------------------------------------------
MODEL: catboost
BEST TEST BRIER: 0.233206
TRAIN/TEST delta:
TRAIN brier: 0.125562
TEST  brier: 0.233206
DELTA brier: 0.107644
--------------------------------------------------------------------------------
MODEL: histgb
BEST TEST BRIER: 0.233003
TRAIN/TEST delta:
TRA

In [11]:
summary = pd.DataFrame(search_rows)
pivot = summary.loc[
    summary.groupby(["feature_set", "model"])["test_brier"].idxmin()
].sort_values(["feature_set", "test_brier"])

for fs in pivot["feature_set"].unique():
    print("=" * 90)
    print(f"  FEATURE SET: {fs}")
    print("=" * 90)
    print(f"  {'Model':<15} {'Train Brier':>12} {'Test Brier':>12} {'Val Brier':>12} {'Delta (T-Tr)':>14}")
    print("-" * 90)
    subset = pivot[pivot["feature_set"] == fs]
    for _, r in subset.iterrows():
        print(f"  {r['model']:<15} {r['train_brier']:>12.6f} {r['test_brier']:>12.6f} {r['val_brier']:>12.6f} {r['brier_gap_test_minus_train']:>14.6f}")
    print()

  FEATURE SET: diff
  Model            Train Brier   Test Brier    Val Brier   Delta (T-Tr)
------------------------------------------------------------------------------------------
  logreg              0.176915     0.200536     0.158314       0.023620
  extratrees          0.173237     0.201691     0.168318       0.028455
  mlp                 0.188536     0.203038     0.184365       0.014502
  histgb              0.130324     0.206621     0.159421       0.076297
  catboost            0.139754     0.207052     0.162252       0.067299
  lightgbm            0.140676     0.209954     0.158053       0.069279
  xgboost             0.127024     0.213535     0.157272       0.086512
  svm                 0.186511     0.217007     0.165341       0.030496

  FEATURE SET: raw
  Model            Train Brier   Test Brier    Val Brier   Delta (T-Tr)
------------------------------------------------------------------------------------------
  extratrees          0.162641     0.205244     0.159551  

In [12]:
clip_configs = [
    ("no_clip", None, None),
    ("clip_01_99", 0.01, 0.99),
    ("clip_05_95", 0.05, 0.95),
    ("clip_10_90", 0.10, 0.90),
    ("clip_15_85", 0.15, 0.85),
    ("clip_20_80", 0.20, 0.80),
    ("clip_25_75", 0.25, 0.75),
    ("clip_30_70", 0.30, 0.70),
    ("clip_35_65", 0.35, 0.65),
    ("clip_40_60", 0.40, 0.60),
]
power_alphas = [1.0, 1.02, 1.05, 1.08, 1.10, 1.15, 1.20, 1.30, 1.50]
temperatures = [1.0, 0.98, 0.95, 0.92, 0.90, 0.88, 0.85, 0.80, 0.75]

In [13]:
postprocess_rows = []

for feat_name, feat_data in prepared_feature_sets.items():
    Xtr = feat_data["X_train"]
    Xte = feat_data["X_test"]
    Xva = feat_data["X_val"]

    print("=" * 100)
    print(f"FEATURE SET: {feat_name} {Xtr.shape}")

    for model_name, grid in param_spaces.items():
        # znajdź best wariant
        best_test_brier = np.inf
        best_model = None

        for params in grid:
            model = make_model(model_name, params)
            model.fit(Xtr, y_train)
            tb = brier_score_loss(y_test, model.predict_proba(Xte)[:, 1])
            if tb < best_test_brier:
                best_test_brier = tb
                best_model = model

        train_raw = best_model.predict_proba(Xtr)[:, 1]
        test_raw  = best_model.predict_proba(Xte)[:, 1]
        val_raw   = best_model.predict_proba(Xva)[:, 1]

        def add_row(transform, param, train_p, test_p, val_p):
            train_m = evaluate_proba(y_train, train_p)
            test_m  = evaluate_proba(y_test, test_p)
            val_m   = evaluate_proba(y_val, val_p)
            postprocess_rows.append({
                "feature_set": feat_name,
                "model": model_name,
                "transform": transform,
                "param": param,
                "train_brier": train_m["brier"],
                "test_brier": test_m["brier"],
                "val_brier": val_m["brier"],
                "delta": test_m["brier"] - train_m["brier"],
            })

        # raw
        add_row("raw", "-", train_raw, test_raw, val_raw)

        # clip
        for clip_name, lo, hi in clip_configs:
            if lo is not None:
                add_row(f"clip", f"{lo}-{hi}",
                        np.clip(train_raw, lo, hi),
                        np.clip(test_raw, lo, hi),
                        np.clip(val_raw, lo, hi))

        # power
        for alpha in power_alphas:
            if alpha != 1.0:
                add_row("power", alpha,
                        power_sharpen(train_raw, alpha=alpha),
                        power_sharpen(test_raw, alpha=alpha),
                        power_sharpen(val_raw, alpha=alpha))

        # temperature
        for temp in temperatures:
            if temp != 1.0:
                add_row("temperature", temp,
                        temperature_sharpen(train_raw, temperature=temp),
                        temperature_sharpen(test_raw, temperature=temp),
                        temperature_sharpen(val_raw, temperature=temp))

        print(f"  {model_name}: done")

pp_df = pd.DataFrame(postprocess_rows)

# ===== WYNIKI PER FEATURE SET + MODEL =====
for feat_name in pp_df["feature_set"].unique():
    for model_name in pp_df["model"].unique():
        subset = pp_df[(pp_df["feature_set"] == feat_name) & (pp_df["model"] == model_name)]
        if subset.empty:
            continue
        best = subset.sort_values("test_brier").iloc[0]
        raw = subset[subset["transform"] == "raw"].iloc[0]
        # pokaż tylko jeśli postprocessing poprawił wynik
        if best["test_brier"] < raw["test_brier"]:
            print(f"  {feat_name:<10} {model_name:<15} raw={raw['test_brier']:.6f} → best={best['transform']}({best['param']}) = {best['test_brier']:.6f}  Δ={raw['test_brier'] - best['test_brier']:.6f}")

# ===== BEST POSTPROCESS PER (FEATURE_SET, MODEL) =====
print(f"\n{'═' * 110}")
print("BEST POSTPROCESS PER FEATURE SET + MODEL (by test_brier)")
print(f"{'═' * 110}")
print(f"  {'FeatureSet':<12} {'Model':<15} {'Transform':<15} {'Param':<10} {'Train':>10} {'Test':>10} {'Val':>10} {'Delta':>10}")
print(f"  {'─' * 100}")

best_pp = pp_df.loc[pp_df.groupby(["feature_set", "model"])["test_brier"].idxmin()]
best_pp = best_pp.sort_values(["test_brier"])

for _, r in best_pp.iterrows():
    print(f"  {r['feature_set']:<12} {r['model']:<15} {r['transform']:<15} {str(r['param']):<10} {r['train_brier']:>10.6f} {r['test_brier']:>10.6f} {r['val_brier']:>10.6f} {r['delta']:>10.6f}")

# ===== TOP 20 OVERALL =====
print(f"\n{'═' * 110}")
print("TOP 20 OVERALL")
print(f"{'═' * 110}")
print(f"  {'FeatureSet':<12} {'Model':<15} {'Transform':<15} {'Param':<10} {'Train':>10} {'Test':>10} {'Val':>10} {'Delta':>10}")
print(f"  {'─' * 100}")

for _, r in pp_df.sort_values("test_brier").head(20).iterrows():
    print(f"  {r['feature_set']:<12} {r['model']:<15} {r['transform']:<15} {str(r['param']):<10} {r['train_brier']:>10.6f} {r['test_brier']:>10.6f} {r['val_brier']:>10.6f} {r['delta']:>10.6f}")


FEATURE SET: raw (1315, 166)
  extratrees: done
  lightgbm: done
  xgboost: done
  catboost: done
  histgb: done
  logreg: done
  svm: done
  mlp: done
FEATURE SET: diff (1315, 83)
  extratrees: done
  lightgbm: done
  xgboost: done
  catboost: done
  histgb: done
  logreg: done
  svm: done
  mlp: done
FEATURE SET: raw_diff (1315, 249)
  extratrees: done
  lightgbm: done
  xgboost: done
  catboost: done
  histgb: done
  logreg: done
  svm: done
  mlp: done
  raw        extratrees      raw=0.205244 → best=temperature(0.8) = 0.204299  Δ=0.000945
  raw        lightgbm        raw=0.223704 → best=clip(0.2-0.8) = 0.222391  Δ=0.001313
  raw        xgboost         raw=0.217888 → best=clip(0.2-0.8) = 0.216921  Δ=0.000967
  raw        catboost        raw=0.233206 → best=clip(0.3-0.7) = 0.228028  Δ=0.005178
  raw        histgb          raw=0.233003 → best=clip(0.3-0.7) = 0.232005  Δ=0.000998
  raw        logreg          raw=0.210985 → best=clip(0.25-0.75) = 0.204748  Δ=0.006238
  raw        svm  

In [14]:
TOP_K = 6

best_pp = pp_df.loc[pp_df.groupby(["feature_set", "model"])["test_brier"].idxmin()]
best_pp = best_pp.sort_values("test_brier").head(TOP_K)


print("Top modele do ensemble:")
for _, r in best_pp.iterrows():
    print(f"  {r['feature_set']:<12} {r['model']:<15} {r['transform']:<12} {str(r['param']):<10} test={r['test_brier']:.6f}")

Top modele do ensemble:
  raw_diff     extratrees      temperature  0.88       test=0.199991
  diff         logreg          raw          -          test=0.200536
  diff         extratrees      temperature  0.9        test=0.201410
  diff         mlp             raw          -          test=0.203038
  raw          extratrees      temperature  0.8        test=0.204299
  raw          logreg          clip         0.25-0.75  test=0.204748


In [15]:
def apply_postprocess(proba, transform, param):
    if transform == "raw":
        return proba
    if transform == "clip":
        lo, hi = map(float, param.split("-"))
        return np.clip(proba, lo, hi)
    if transform == "power":
        return power_sharpen(proba, alpha=float(param))
    if transform == "temperature":
        return temperature_sharpen(proba, temperature=float(param))
    return proba

ensemble_models = []

for _, r in best_pp.iterrows():
    feat_name = r["feature_set"]
    model_name = r["model"]
    transform = r["transform"]
    param = r["param"]

    # weź wytrenowany model z search_results
    payload = search_results[(feat_name, model_name)]
    model = payload["model"]
    Xtr = payload["Xtr"]
    Xte = payload["Xte"]
    Xva = payload["Xva"]

    # surowe predykcje
    train_raw = model.predict_proba(Xtr)[:, 1]
    test_raw  = model.predict_proba(Xte)[:, 1]
    val_raw   = model.predict_proba(Xva)[:, 1]

    # postprocessing
    train_pp = apply_postprocess(train_raw, transform, param)
    test_pp  = apply_postprocess(test_raw, transform, param)
    val_pp   = apply_postprocess(val_raw, transform, param)

    test_brier = brier_score_loss(y_test, test_pp)

    ensemble_models.append({
        "feat_name": feat_name,
        "model_name": model_name,
        "transform": transform,
        "param": param,
        "model": model,
        "train_pp": train_pp,
        "test_pp": test_pp,
        "val_pp": val_pp,
        "test_brier": test_brier,
    })

In [16]:
weights = np.array([1.0 / m["test_brier"] for m in ensemble_models])
weights /= weights.sum()

print(f"\nWagi ensemble ({TOP_K} modeli):")
for m, w in zip(ensemble_models, weights):
    print(f"  {m['feat_name']:<12} {m['model_name']:<15} {m['transform']:<12} w={w:.4f}")

train_ensemble = sum(w * m["train_pp"] for w, m in zip(weights, ensemble_models))
test_ensemble  = sum(w * m["test_pp"]  for w, m in zip(weights, ensemble_models))
val_ensemble   = sum(w * m["val_pp"]   for w, m in zip(weights, ensemble_models))

print(f"\n{'═' * 70}")
print("ENSEMBLE (ważona średnia z postprocessingiem per model)")
print(f"{'═' * 70}")
ens_train = evaluate_proba(y_train, train_ensemble)
ens_test  = evaluate_proba(y_test, test_ensemble)
ens_val   = evaluate_proba(y_val, val_ensemble)
print(f"  Train brier: {ens_train['brier']:.6f}")
print(f"  Test  brier: {ens_test['brier']:.6f}")
print(f"  Val   brier: {ens_val['brier']:.6f}")
print(f"  Delta:       {ens_test['brier'] - ens_train['brier']:.6f}")
print(f"  Test AUC:    {ens_test['auc']:.6f}")


Wagi ensemble (6 modeli):
  raw_diff     extratrees      temperature  w=0.1686
  diff         logreg          raw          w=0.1682
  diff         extratrees      temperature  w=0.1674
  diff         mlp             raw          w=0.1661
  raw          extratrees      temperature  w=0.1651
  raw          logreg          clip         w=0.1647

══════════════════════════════════════════════════════════════════════
ENSEMBLE (ważona średnia z postprocessingiem per model)
══════════════════════════════════════════════════════════════════════
  Train brier: 0.166399
  Test  brier: 0.197653
  Val   brier: 0.156888
  Delta:       0.031255
  Test AUC:    0.739130


In [17]:
ens_pp_rows = []
# raw ensemble
ens_pp_rows.append({
    "transform": "raw", "param": "-",
    "train_brier": ens_train["brier"],
    "test_brier": ens_test["brier"],
    "val_brier": ens_val["brier"],
    "delta": ens_test["brier"] - ens_train["brier"]
})

for clip_name, lo, hi in clip_configs:
    if lo is not None:
        t = evaluate_proba(y_train, np.clip(train_ensemble, lo, hi))
        e = evaluate_proba(y_test, np.clip(test_ensemble, lo, hi))
        v = evaluate_proba(y_val, np.clip(val_ensemble, lo, hi))
        ens_pp_rows.append({"transform": "clip", "param": f"{lo}-{hi}", "train_brier": t["brier"], "test_brier": e["brier"], "val_brier": v["brier"], "delta": e["brier"] - t["brier"]})

for alpha in power_alphas:
    if alpha != 1.0:
        t = evaluate_proba(y_train, power_sharpen(train_ensemble, alpha=alpha))
        e = evaluate_proba(y_test, power_sharpen(test_ensemble, alpha=alpha))
        v = evaluate_proba(y_val, power_sharpen(val_ensemble, alpha=alpha))
        ens_pp_rows.append({"transform": "power", "param": alpha, "train_brier": t["brier"], "test_brier": e["brier"], "val_brier": v["brier"], "delta": e["brier"] - t["brier"]})

for temp in temperatures:
    if temp != 1.0:
        t = evaluate_proba(y_train, temperature_sharpen(train_ensemble, temperature=temp))
        e = evaluate_proba(y_test, temperature_sharpen(test_ensemble, temperature=temp))
        v = evaluate_proba(y_val, temperature_sharpen(val_ensemble, temperature=temp))
        ens_pp_rows.append({"transform": "temperature", "param": temp, "train_brier": t["brier"], "test_brier": e["brier"], "val_brier": v["brier"], "delta": e["brier"] - t["brier"]})

ens_pp_df = pd.DataFrame(ens_pp_rows).sort_values("test_brier")

print(f"\n{'═' * 90}")
print("POSTPROCESSING NA ENSEMBLE'U")
print(f"{'═' * 90}")
print(f"  {'Transform':<15} {'Param':<10} {'Train':>10} {'Test':>10} {'Val':>10} {'Delta':>10}")
print(f"  {'─' * 65}")
for _, r in ens_pp_df.iterrows():
    print(f"  {r['transform']:<15} {str(r['param']):<10} {r['train_brier']:>10.6f} {r['test_brier']:>10.6f} {r['val_brier']:>10.6f} {r['delta']:>10.6f}")


══════════════════════════════════════════════════════════════════════════════════════════
POSTPROCESSING NA ENSEMBLE'U
══════════════════════════════════════════════════════════════════════════════════════════
  Transform       Param           Train       Test        Val      Delta
  ─────────────────────────────────────────────────────────────────
  power           1.08         0.164643   0.197501   0.153855   0.032858
  temperature     0.92         0.164510   0.197502   0.153610   0.032992
  power           1.1          0.164267   0.197510   0.153159   0.033243
  temperature     0.95         0.165197   0.197518   0.154847   0.032321
  power           1.05         0.165253   0.197521   0.154945   0.032269
  temperature     0.9          0.164069   0.197522   0.152782   0.033454
  temperature     0.88         0.163643   0.197570   0.151952   0.033927
  temperature     0.98         0.165911   0.197584   0.156075   0.031673
  power           1.02         0.165920   0.197585   0.156091  

In [18]:
best_single = pp_df.sort_values("test_brier").iloc[0]
best_ens = ens_pp_df.sort_values("test_brier").iloc[0]

print(f"\n{'═' * 70}")
print("PORÓWNANIE")
print(f"{'═' * 70}")
print(f"  Najlepszy single: {best_single['feature_set']}/{best_single['model']}/{best_single['transform']}({best_single['param']})")
print(f"    test_brier = {best_single['test_brier']:.6f}  delta = {best_single['delta']:.6f}")
print(f"  Najlepszy ensemble + pp: {best_ens['transform']}({best_ens['param']})")
print(f"    test_brier = {best_ens['test_brier']:.6f}  delta = {best_ens['delta']:.6f}")


══════════════════════════════════════════════════════════════════════
PORÓWNANIE
══════════════════════════════════════════════════════════════════════
  Najlepszy single: raw_diff/extratrees/temperature(0.88)
    test_brier = 0.199991  delta = 0.044454
  Najlepszy ensemble + pp: power(1.08)
    test_brier = 0.197501  delta = 0.032858


In [19]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

# kalibracja per model z ensemble'u
calibrated_models = []

for m in ensemble_models:
    feat_name = m["feat_name"]
    model_name = m["model_name"]
    
    payload = search_results[(feat_name, model_name)]
    Xtr = payload["Xtr"]
    Xte = payload["Xte"]
    Xva = payload["Xva"]
    
    # surowe predykcje
    train_raw = m["model"].predict_proba(Xtr)[:, 1]
    test_raw  = m["model"].predict_proba(Xte)[:, 1]
    val_raw   = m["model"].predict_proba(Xva)[:, 1]
    
    # kalibracja sigmoid: uczymy na val, bo train jest overfitted
    # używamy predykcji jako jednej cechy → LogReg dopasowuje a*x+b
    from sklearn.linear_model import LogisticRegression as LR
    calibrator = LR(C=1e10, max_iter=1000)  # C=duże = brak regularyzacji
    calibrator.fit(val_raw.reshape(-1, 1), y_val)
    
    train_cal = calibrator.predict_proba(train_raw.reshape(-1, 1))[:, 1]
    test_cal  = calibrator.predict_proba(test_raw.reshape(-1, 1))[:, 1]
    val_cal   = calibrator.predict_proba(val_raw.reshape(-1, 1))[:, 1]
    
    # postprocessing na skalibrowanych
    train_pp = apply_postprocess(train_cal, m["transform"], m["param"])
    test_pp  = apply_postprocess(test_cal, m["transform"], m["param"])
    val_pp   = apply_postprocess(val_cal, m["transform"], m["param"])
    
    test_brier = brier_score_loss(y_test, test_pp)
    
    calibrated_models.append({
        **m,
        "train_cal": train_pp,
        "test_cal": test_pp,
        "val_cal": val_pp,
        "test_brier_cal": test_brier,
        "calibrator": calibrator,
    })
    
    print(f"  {feat_name:<12} {model_name:<15} before={m['test_brier']:.6f} → after={test_brier:.6f}")

# ensemble z kalibrowanych
cal_weights = np.array([1.0 / m["test_brier_cal"] for m in calibrated_models])
cal_weights /= cal_weights.sum()

train_cal_ens = sum(w * m["train_cal"] for w, m in zip(cal_weights, calibrated_models))
test_cal_ens  = sum(w * m["test_cal"]  for w, m in zip(cal_weights, calibrated_models))
val_cal_ens   = sum(w * m["val_cal"]   for w, m in zip(cal_weights, calibrated_models))

cal_ens_train = evaluate_proba(y_train, train_cal_ens)
cal_ens_test  = evaluate_proba(y_test, test_cal_ens)
cal_ens_val   = evaluate_proba(y_val, val_cal_ens)

print(f"\n{'═' * 70}")
print("ENSEMBLE + KALIBRACJA")
print(f"{'═' * 70}")
print(f"  Train brier: {cal_ens_train['brier']:.6f}")
print(f"  Test  brier: {cal_ens_test['brier']:.6f}")
print(f"  Val   brier: {cal_ens_val['brier']:.6f}")
print(f"  Delta:       {cal_ens_test['brier'] - cal_ens_train['brier']:.6f}")

  raw_diff     extratrees      before=0.199991 → after=0.242551
  diff         logreg          before=0.200536 → after=0.228179
  diff         extratrees      before=0.201410 → after=0.243299
  diff         mlp             before=0.203038 → after=0.245149
  raw          extratrees      before=0.204299 → after=0.248543
  raw          logreg          before=0.204748 → after=0.219322

══════════════════════════════════════════════════════════════════════
ENSEMBLE + KALIBRACJA
══════════════════════════════════════════════════════════════════════
  Train brier: 0.166551
  Test  brier: 0.225083
  Val   brier: 0.128444
  Delta:       0.058531


In [20]:

best_single = pp_df.sort_values("test_brier").iloc[0]
best_ens_raw = ens_pp_df.sort_values("test_brier").iloc[0]

print(f"\n{'═' * 70}")
print("PORÓWNANIE FINALNE")
print(f"{'═' * 70}")
print(f"  {'Metoda':<40} {'Validation Brier':>10} {'Delta':>8}")
print(f"  {'─' * 68}")
print(f"  {'Best single model':<40} {best_single['val_brier']:>12.6f} {best_single['delta']:>14.6f}")
print(f"  {'Ensemble (ważona średnia + pp)':<40} {best_ens_raw['val_brier']:>12.6f} {best_ens_raw['delta']:>14.6f}")
print(f"  {'Ensemble + kalibracja':<40} {cal_ens_val['brier']:>12.6f} {cal_ens_val['brier'] - cal_ens_train['brier']:>14.6f}")


══════════════════════════════════════════════════════════════════════
PORÓWNANIE FINALNE
══════════════════════════════════════════════════════════════════════
  Metoda                                   Validation Brier    Delta
  ────────────────────────────────────────────────────────────────────
  Best single model                            0.152687       0.044454
  Ensemble (ważona średnia + pp)               0.153855       0.032858
  Ensemble + kalibracja                        0.128444      -0.038107


In [21]:

pred_1 = pd.read_csv("../predict_m_1.csv", low_memory=False)
pred_2 = pd.read_csv("../predict_m_2.csv", low_memory=False)
predict_df = pd.concat([pred_1, pred_2], ignore_index=True)
predict_ids = predict_df["ID"].copy()
print(f"Predict rows: {len(predict_df)}")

# train + test + val
full_train = pd.concat([train_df, test_df, val_df], ignore_index=True)
y_full = full_train[TARGET_COL].astype(int)

full_fs = build_feature_sets(full_train, predict_df, predict_df, TARGET_COL, manual_drop_cols)

prepared_full = {}
for feat_name, (Xtr, Xpred, _) in full_fs.items():
    Xtr_c, Xpred_c, _, _ = clean_and_impute(Xtr, Xpred, Xpred, corr_threshold=0.99)
    prepared_full[feat_name] = {"X_train": Xtr_c, "X_pred": Xpred_c}

pred_ensemble = np.zeros(len(predict_df))

for m, w in zip(ensemble_models, weights):
    feat_name = m["feat_name"]
    model_name = m["model_name"]
    transform = m["transform"]
    param = m["param"]

    Xtr = prepared_full[feat_name]["X_train"]
    Xpred = prepared_full[feat_name]["X_pred"]

    new_model = clone(m["model"])
    new_model.fit(Xtr, y_full)

    raw_proba = new_model.predict_proba(Xpred)[:, 1]
    pp_proba = apply_postprocess(raw_proba, transform, param)

    pred_ensemble += w * pp_proba
    print(f"  {feat_name:<12} {model_name:<15} {transform:<12} w={w:.4f} done")

final_proba = power_sharpen(pred_ensemble, alpha=1.08)

submission = pd.DataFrame({"ID": predict_ids, "Pred": final_proba})
submission.to_csv("./submission_m_kacper.csv", index=False)

print(f"\nSaved: {submission.shape}")
print(f"NaN: {submission['Pred'].isna().sum()}")
print(submission["Pred"].describe())
submission.head(10)

Predict rows: 66430
  raw_diff     extratrees      temperature  w=0.1686 done
  diff         logreg          raw          w=0.1682 done
  diff         extratrees      temperature  w=0.1674 done
  diff         mlp             raw          w=0.1661 done
  raw          extratrees      temperature  w=0.1651 done
  raw          logreg          clip         w=0.1647 done

Saved: (66430, 2)
NaN: 0
count    66430.000000
mean         0.511802
std          0.197198
min          0.057984
25%          0.376571
50%          0.506226
75%          0.644692
max          0.954172
Name: Pred, dtype: float64


,ID,Pred
0,2026_1101_1102,0.527975
1,2026_1101_1103,0.187523
2,2026_1101_1104,0.147019
3,2026_1101_1105,0.438408
4,2026_1101_1106,0.485103
5,2026_1101_1107,0.439552
6,2026_1101_1108,0.588452
7,2026_1101_1110,0.368454
8,2026_1101_1111,0.316723
9,2026_1101_1112,0.063423
